# Lecture 3 — Class Exercise
## Line Charts: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
- No spaghetti — if you show multiple lines, grey everything and highlight one
- Remove chart borders, heavy gridlines, rotating text
- Label directly; avoid legends where possible
- Insight title


In [15]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
df = pd.read_csv('owid-co2-data.csv')

# Check actual columns and map them safely
print('Available columns:', [c for c in df.columns if 'continent' in c or 'country' in c or 'co2' in c])

# Map standard OWID columns
rename_dict = {'country': 'Country', 'year': 'Year', 'co2': 'CO2_Mt', 'co2_per_capita': 'CO2_per_capita'}
if 'continent' in df.columns:
    rename_dict['continent'] = 'Region'
elif 'region' in df.columns:
    rename_dict['region'] = 'Region'

df = df.rename(columns=rename_dict)

# Filter for the 2000-2022 range
df = df[(df['Year'] >= 2000) & (df['Year'] <= 2022)]

# Keep only rows that have a Region/Continent and necessary CO2 data
if 'Region' in df.columns:
    df = df.dropna(subset=['Region', 'CO2_Mt', 'CO2_per_capita'])
else:
    # Fallback if no region column: filter out rows with null iso_code (non-countries)
    if 'iso_code' in df.columns:
        df = df.dropna(subset=['iso_code'])
    df = df.dropna(subset=['CO2_Mt', 'CO2_per_capita'])
    df['Region'] = 'Global' # fallback

print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())

Available columns: ['country', 'cement_co2', 'cement_co2_per_capita', 'co2', 'co2_growth_abs', 'co2_growth_prct', 'co2_including_luc', 'co2_including_luc_growth_abs', 'co2_including_luc_growth_prct', 'co2_including_luc_per_capita', 'co2_including_luc_per_gdp', 'co2_including_luc_per_unit_energy', 'co2_per_capita', 'co2_per_gdp', 'co2_per_unit_energy', 'coal_co2', 'coal_co2_per_capita', 'consumption_co2', 'consumption_co2_per_capita', 'consumption_co2_per_gdp', 'cumulative_cement_co2', 'cumulative_co2', 'cumulative_co2_including_luc', 'cumulative_coal_co2', 'cumulative_flaring_co2', 'cumulative_gas_co2', 'cumulative_luc_co2', 'cumulative_oil_co2', 'cumulative_other_co2', 'flaring_co2', 'flaring_co2_per_capita', 'gas_co2', 'gas_co2_per_capita', 'land_use_change_co2', 'land_use_change_co2_per_capita', 'oil_co2', 'oil_co2_per_capita', 'other_co2_per_capita', 'other_industry_co2', 'share_global_cement_co2', 'share_global_co2', 'share_global_co2_including_luc', 'share_global_coal_co2', 'shar

In [16]:
# Explore before building
print('Countries:', df['Country'].unique())
print('\nCO2 range:', df['CO2_Mt'].min(), 'to', df['CO2_Mt'].max(), 'Mt')
print('\nRegional averages (2022):')
print(
    df[df['Year'] == 2022]
    .groupby('Region')['CO2_Mt']
    .mean()
    .sort_values(ascending=False)
    .round(1)
)

Countries: ['Afghanistan' 'Albania' 'Algeria' 'Andorra' 'Angola' 'Anguilla'
 'Antigua and Barbuda' 'Argentina' 'Armenia' 'Aruba' 'Australia' 'Austria'
 'Azerbaijan' 'Bahamas' 'Bahrain' 'Bangladesh' 'Barbados' 'Belarus'
 'Belgium' 'Belize' 'Benin' 'Bermuda' 'Bhutan' 'Bolivia'
 'Bonaire Sint Eustatius and Saba' 'Bosnia and Herzegovina' 'Botswana'
 'Brazil' 'British Virgin Islands' 'Brunei' 'Bulgaria' 'Burkina Faso'
 'Burundi' 'Cambodia' 'Cameroon' 'Canada' 'Cape Verde'
 'Central African Republic' 'Chad' 'Chile' 'China' 'Colombia' 'Comoros'
 'Congo' 'Cook Islands' 'Costa Rica' "Cote d'Ivoire" 'Croatia' 'Cuba'
 'Curacao' 'Cyprus' 'Czechia' 'Democratic Republic of Congo' 'Denmark'
 'Djibouti' 'Dominica' 'Dominican Republic' 'East Timor' 'Ecuador' 'Egypt'
 'El Salvador' 'Equatorial Guinea' 'Eritrea' 'Estonia' 'Eswatini'
 'Ethiopia' 'Faroe Islands' 'Fiji' 'Finland' 'France' 'French Polynesia'
 'Gabon' 'Gambia' 'Georgia' 'Germany' 'Ghana' 'Greece' 'Greenland'
 'Grenada' 'Guatemala' 'Guinea' 'G

## Task 1 - A line chart showing CO2 per Capita for all countries in the dataset

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#EEEEEE), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

In [20]:
# Task 1 - A line chart showing CO2 per Capita for all countries in the dataset
import plotly.graph_objects as go

highlight_country = 'United States'

fig1 = go.Figure()

for country in df['Country'].unique():
  df_country = df[df['Country'] == country]

  if country == highlight_country:
    # Highlighted country (colored & thicker)
    fig1.add_trace(
        go.Scatter(
            x=df_country['Year'],
            y=df_country['CO2_per_capita'],
            mode='lines',
            name=country,
            line=dict(color='#1f77b4', width=3),
        )
    )
    # Direct label at the end of the line
    latest_row = df_country[df_country['Year'] == df_country['Year'].max()]
    fig1.add_annotation(
        x=latest_row['Year'].values[0],
        y=latest_row['CO2_per_capita'].values[0],
        text=f'  {country}',
        showarrow=False,
        xanchor='left',
        font=dict(color='#1f77b4', size=12, family='sans-serif'),
    )
  else:
    # Background context lines (thin grey)
    fig1.add_trace(
        go.Scatter(
            x=df_country['Year'],
            y=df_country['CO2_per_capita'],
            mode='lines',
            name=country,
            line=dict(color='#d3d3d3', width=1),
            hoverinfo='skip',
        )
    )

fig1.update_layout(
    title=(
        '<b>United States Maintains the Highest Per Capita CO2 Emissions'
        ' Despite Steady Reductions (2000–2022)</b>'
    ),
    xaxis_title='Year',
    yaxis_title='CO2 per Capita (t)',
    showlegend=False,
    template='simple_white',
    margin=dict(r=130),
)

fig1.show()

---
## Task 2 — A line chart showing CO2 emissions over time for all Asian countries in the dataset, with one country highlighted.

**same requirements as Task1**

In [22]:
# Task 2 — A line chart showing CO2 emissions over time for all Asian countries in the dataset
import plotly.graph_objects as go

# Filter for Asian countries (using 'Region' or 'continent' column if available, or filtering by known Asian countries)
# If Region column exists:
if 'Region' in df.columns:
  asian_df = df[df['Region'] == 'Asia']
else:
  # Fallback list of major Asian countries if Region isn't mapped
  asian_countries = [
      'China',
      'India',
      'Japan',
      'South Korea',
      'Indonesia',
      'Saudi Arabia',
      'Iran',
      'Pakistan',
      'Thailand',
      'Malaysia',
  ]
  asian_df = df[df['Country'].isin(asian_countries)]

highlight_asian = 'China'

fig2 = go.Figure()

for country in asian_df['Country'].unique():
  df_country = asian_df[asian_df['Country'] == country]

  if country == highlight_asian:
    # Highlighted country (colored & thicker)
    fig2.add_trace(
        go.Scatter(
            x=df_country['Year'],
            y=df_country['CO2_Mt'],
            mode='lines',
            name=country,
            line=dict(color='#d62728', width=3),
        )
    )
    # Direct label at the end of the line
    latest_row = df_country[df_country['Year'] == df_country['Year'].max()]
    fig2.add_annotation(
        x=latest_row['Year'].values[0],
        y=latest_row['CO2_Mt'].values[0],
        text=f'  {country}',
        showarrow=False,
        xanchor='left',
        font=dict(color='#d62728', size=12, family='sans-serif'),
    )
  else:
    # Background context lines (thin grey)
    fig2.add_trace(
        go.Scatter(
            x=df_country['Year'],
            y=df_country['CO2_Mt'],
            mode='lines',
            name=country,
            line=dict(color='#e0e0e0', width=1),
            hoverinfo='skip',
        )
    )

fig2.update_layout(
    title=(
        "<b>China's Total CO2 Emissions Surged Dramatically Between 2000 and"
        ' 2022 Relative to Other Asian Nations</b>'
    ),
    xaxis_title='Year',
    yaxis_title='CO2 Emissions (Mt)',
    showlegend=False,
    template='simple_white',
    margin=dict(r=130),
)

fig2.show()
